In [24]:
import pandas as pd
""
df = pd.read_csv('forest_reserve_state.csv')

df.head()

,date,state,area
0,2003-01-01,Johor,356922.0
1,2003-01-01,Kedah,344530.0
2,2003-01-01,Kelantan,629687.0
3,2003-01-01,Melaka,5468.0
4,2003-01-01,Negeri Sembilan,165639.0


In [25]:
# Convert date column to datetime & extract year
df['year'] = pd.to_datetime(df['date']).dt.year

# 🔹 Clean data first
df = df[df['state'] != 'Semenanjung Malaysia']

rename_map = {
    'W.P. Kuala Lumpur': 'Kuala Lumpur',
    'W.P. Labuan': 'Labuan',
    'W.P. Putrajaya': 'Putrajaya'
}
df['state'] = df['state'].replace(rename_map)

# (Optional) Reset index
df = df.reset_index(drop=True)

# 🔹 Now find latest year
latest_year = df['year'].max()

# 🔹 Filter for the latest year
df_latest = df[df['year'] == latest_year]

print("Latest year:", latest_year)
print(df_latest.head())

Latest year: 2021
           date            state      area  year
288  2021-01-01            Johor  334502.0  2021
289  2021-01-01            Kedah  341976.0  2021
290  2021-01-01         Kelantan  629881.0  2021
291  2021-01-01           Melaka    5199.0  2021
292  2021-01-01  Negeri Sembilan  155143.0  2021


In [26]:
df_latest.to_csv("forest_reserve_cleaned.csv", index=False)

In [1]:
import pandas as pd 

df = pd.read_csv("air_pollution.csv", parse_dates=["date"])

print(df.head())

        date pollutant  concentration
0 2017-01-01        CO          0.561
1 2017-02-01        CO          0.530
2 2017-03-01        CO          0.589
3 2017-04-01        CO          0.662
4 2017-05-01        CO            NaN


In [2]:
# Extract year and month
df['year'] = df['date'].dt.year

# make sure concentration is numeric (empty strings -> NaN)
df['concentration'] = pd.to_numeric(df['concentration'], errors='coerce')

df = df[df['year'].isin([2019, 2020, 2021])]

# drop missing concentration values 
df = df.dropna(subset=['concentration'])

# group by pollutant + year, compute average
agg = df.groupby(["pollutant", "year"], as_index=False)["concentration"].mean()

# build hierarchical structure
rows = []
rows.append({'id': 'All', 'parent': '', 'value': ''})   # root -> parent empty cell

for pollutant in sorted(agg['pollutant'].unique()):
    rows.append({'id': pollutant, 'parent': 'All', 'value': ''})
    subset = agg[agg['pollutant'] == pollutant].sort_values('year')
    for _, r in subset.iterrows():
        rows.append({
            'id': f"{pollutant}-{int(r['year'])}",
            'parent': pollutant,
            'value': round(float(r['concentration']), 6)
        })

out = pd.DataFrame(rows, columns=['id','parent','value'])

In [3]:
out.head(30)

,id,parent,value
0,All,,
1,CO,All,
2,CO-2019,CO,0.650083
3,CO-2020,CO,0.544917
4,CO-2021,CO,0.53375
5,NO2,All,
6,NO2-2019,NO2,0.007192
7,NO2-2020,NO2,0.005767
8,NO2-2021,NO2,0.005692
9,O3,All,


In [4]:
out.to_csv("pollutants_cleaned.csv", index=False)
print("Cleaned file saved as pollutants_cleaned.csv")

Cleaned file saved as pollutants_cleaned.csv


In [2]:
import pandas as pd

# Load rainfall dataset
rainfall = pd.read_csv("mean_temp_rainfall.csv")

# Load pollution dataset
pollution = pd.read_csv("air_pollution.csv")

print(rainfall.head())
print(pollution.head())

      State Selected meteorological station   \
0     Johor                            Senai   
1     Johor                           Kluang   
2     Kedah                       Alor Setar   
3     Kedah                   Pulau Langkawi   
4  Kelantan                       Kota Bharu   

  Height above mean sea level in metres  Year  \
0                               (37.8m)  2000   
1                               (88.1m)  2000   
2                                (3.9m)  2000   
3                                (6.4m)  2000   
4                                (4.4m)  2000   

  Minimum Mean temperature in Celcius Maximum Mean temperature in Celcius  \
0                                22.9                                32.3   
1                                23.1                                32.0   
2                                23.6                                32.5   
3                                25.0                                32.0   
4                              

In [4]:
### rainfall data
# Force rainfall to numeric (remove text, commas, etc. if any)
rainfall["Total Rainfall in millimetres"] = pd.to_numeric(
    rainfall["Total Rainfall in millimetres"], errors="coerce"
)

# Keep relevant columns
rainfall_clean = rainfall[["State", "Year", "Total Rainfall in millimetres"]]

# Average rainfall per state per year (since some states have multiple stations)
rainfall_yearly = rainfall_clean.groupby(["State", "Year"], as_index=False).mean(numeric_only=True)

rainfall_yearly.rename(columns={"Total Rainfall in millimetres": "Rainfall_mm"}, inplace=True)

### pollution data
# Extract year from date
pollution["Year"] = pd.to_datetime(pollution["date"]).dt.year

# Remove 2017
pollution = pollution[pollution["Year"] != 2017]

# Average pollution per year
pollution_yearly = pollution.groupby("Year", as_index=False)["concentration"].mean()

# Merge by year
merged = pd.merge(rainfall_yearly, pollution_yearly, on="Year", how="inner")

merged.head(20)

,State,Year,Rainfall_mm,concentration
0,Johor,2018,2266.525000,6.974543
1,Johor,2019,2174.525000,8.387444
2,Johor,2020,2458.800000,5.431035
3,Johor,2021,2635.200000,5.668204
4,Kedah,2018,2202.200000,6.974543
5,Kedah,2019,2296.950000,8.387444
6,Kedah,2020,2570.000000,5.431035
7,Kedah,2021,2272.000000,5.668204
8,Kelantan,2018,2592.100000,6.974543
9,Kelantan,2019,1758.066667,8.387444


In [5]:
# Create categories based on tertiles
merged["PollutionCategory"] = pd.qcut(merged["concentration"], q=3, labels=["Low", "Medium", "High"])

merged.head(20)

,State,Year,Rainfall_mm,concentration,PollutionCategory
0,Johor,2018,2266.525000,6.974543,Medium
1,Johor,2019,2174.525000,8.387444,High
2,Johor,2020,2458.800000,5.431035,Low
3,Johor,2021,2635.200000,5.668204,Low
4,Kedah,2018,2202.200000,6.974543,Medium
5,Kedah,2019,2296.950000,8.387444,High
6,Kedah,2020,2570.000000,5.431035,Low
7,Kedah,2021,2272.000000,5.668204,Low
8,Kelantan,2018,2592.100000,6.974543,Medium
9,Kelantan,2019,1758.066667,8.387444,High


In [6]:
merged.to_csv("rainfall_pollution_ready.csv", index=False)

In [14]:
import pandas as pd 

pollution = pd.read_csv("air_pollution_station.csv")
rainfall = pd.read_csv("mean_temp_rainfall.csv")

# --- Clean column names ---
pollution.columns = pollution.columns.str.strip()
rainfall.columns = rainfall.columns.str.strip()

# --- Convert numeric columns (this part comes BEFORE any groupby or merge) ---
pollution["Maximum"] = pd.to_numeric(pollution["Maximum"], errors="coerce")
pollution["Minimum"] = pd.to_numeric(pollution["Minimum"], errors="coerce")

rainfall["Total Rainfall in millimetres"] = pd.to_numeric(rainfall["Total Rainfall in millimetres"], errors="coerce")
rainfall["Number of Days of Rainfall"] = pd.to_numeric(rainfall["Number of Days of Rainfall"], errors="coerce")
rainfall["Mean relative humidity in Percentage"] = pd.to_numeric(rainfall["Mean relative humidity in Percentage"], errors="coerce")
rainfall["Maximum Mean temperature in Celcius"] = pd.to_numeric(rainfall["Maximum Mean temperature in Celcius"], errors="coerce")
rainfall["Minimum Mean temperature in Celcius"] = pd.to_numeric(rainfall["Minimum Mean temperature in Celcius"], errors="coerce")

# --- Compute average pollution per record ---
pollution["AvgPollution"] = (pollution["Maximum"] + pollution["Minimum"]) / 2

# --- If you have multiple stations per state, average them ---
pollution_state = pollution.groupby(["Year", "State"], as_index=False)["AvgPollution"].mean()

# --- Average rainfall per state per year ---
rainfall_summary = rainfall.groupby(["Year", "State"], as_index=False).agg({
    "Total Rainfall in millimetres": "mean",
    "Number of Days of Rainfall": "mean"
})
rainfall_summary.rename(columns={
    "Total Rainfall in millimetres": "Rainfall_mm",
    "Number of Days of Rainfall": "RainDays"
}, inplace=True)

# --- Merge pollution + rainfall ---
merged = pd.merge(pollution_state, rainfall_summary, on=["Year", "State"], how="inner")

# --- Export to CSV for Vega-Lite ---
merged.to_csv("pollution_vs_rainfall.csv", index=False)
print(merged.head(10))


KeyError: 'State'

In [1]:
import pandas as pd

# Load your raw data (replace filename with your actual path)
df = pd.read_csv("tree_cover_loss_by_driver.csv")

# Show a few rows
df.head()

,drivers_type,loss_year,loss_area_ha,gross_carbon_emissions_Mg
0,Hard commodities,2001,2242.427682,1.210344e+06
1,Logging,2001,49884.599047,3.083325e+07
2,Other natural disturbances,2001,368.290169,2.240721e+05
3,Permanent agriculture,2001,267004.924729,9.340107e+07
4,Settlements & Infrastructure,2001,5756.636416,2.563977e+06


In [5]:
df = df.rename(columns={
    "drivers_type": "driver_type",
    "loss_year": "year",
    "loss_area_ha": "loss_area_ha",
    "gross_carbon_emissions_Mg": "carbon_emission_Mg"
})

minor_types = ["Hard commodities", "Other natural disturbances", "Unknown"]

df["driver_type"] = df["driver_type"].replace(minor_types, "Other")

df = df.groupby(["driver_type", "year"], as_index=False).agg({
    "loss_area_ha": "sum",
    "carbon_emission_Mg": "sum"
})

main_drivers = [
    "Permanent agriculture",
    "Logging",
    "Shifting cultivation",
    "Settlements & Infrastructure",
    "Wildfire",
    "Other"
]

# Keep only the main ones
df = df[df["driver_type"].isin(main_drivers)]

df["year"] = df["year"].astype(int)
df["loss_area_ha"] = df["loss_area_ha"].astype(float)
df["carbon_emission_Mg"] = df["carbon_emission_Mg"].astype(float)

df.info()
df.to_csv("tree_loss_clean.csv", index=False)
print("Clean dataset saved as tree_loss_clean.csv")
df.head(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   driver_type         144 non-null    object 
 1   year                144 non-null    int64  
 2   loss_area_ha        144 non-null    float64
 3   carbon_emission_Mg  144 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 4.6+ KB
Clean dataset saved as tree_loss_clean.csv


,driver_type,year,loss_area_ha,carbon_emission_Mg
0,Logging,2001,49884.599047,3.083325e+07
1,Logging,2002,36866.309788,2.442155e+07
2,Logging,2003,34998.463405,2.545375e+07
3,Logging,2004,66820.315935,5.130144e+07
4,Logging,2005,54901.248742,4.300234e+07
5,Logging,2006,57025.864602,4.335688e+07
6,Logging,2007,80691.037935,6.265439e+07
7,Logging,2008,73625.258540,5.827702e+07
8,Logging,2009,97086.859639,7.642175e+07
9,Logging,2010,59684.214713,4.657186e+07
